<center>
<a href="https://www.umontpellier.fr/"><img src="https://www.umontpellier.fr/wp-content/uploads/2022/10/logo_um_2022_rouge_rvb.svg" width="200"/></a>&nbsp;&nbsp;
<a href="https://economie.edu.umontpellier.fr/"><img src="https://economie.edu.umontpellier.fr/files/2014/12/economie_rvb_2015-300x137.png" width="160"/></a>
</center>

<div align="center">

#  Phase 2 — Nettoyage et Validation de la Base `ressources.csv`

| Nom et Prénom | Rôle |
|---|---|
| Randriamisaina Tsiory-Fanomezana | Membre de l'équipe |
| SHIRALI POUR Amir | Membre de l'équipe |

</div>

---
## Objectif de cette phase

La base `ressources.csv` est le **journal de présence quotidienne** des agents.  
Chaque ligne correspond à une journée de présence d'un agent avec ses caractéristiques contractuelles.

Ce notebook applique toutes les corrections décrites dans [`docs/phase2_ressources.md`](../docs/phase2_ressources.md).

**Démarche pour chaque anomalie :**
1. **Constat** — quantification du problème
2. **Décision** — avec justification métier/statistique
3. **Action** — code de correction
4. **Vérification** — preuve que la correction est effective


---
## Section 0 — Initialisation

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# --- Localisation de la racine du projet
def localiser_racine_du_projet():
    """Remonte l'arborescence jusqu'à trouver la racine (présence de .git ou requirements.txt)."""
    repertoire_courant = Path.cwd()
    while True:
        if any((repertoire_courant / m).exists() for m in ['.git', 'requirements.txt']):
            break
        if repertoire_courant.parent == repertoire_courant:
            break
        repertoire_courant = repertoire_courant.parent
    if Path.cwd() != repertoire_courant:
        os.chdir(repertoire_courant)
    return repertoire_courant.resolve()

REPERTOIRE_RACINE = localiser_racine_du_projet()
sys.path.insert(0, str(REPERTOIRE_RACINE / 'src'))

from utils.dataframe_styler import style_duplicates
print(f" Racine du projet : {REPERTOIRE_RACINE}")

In [ ]:
# --- Chargement de la base ressources
ressources_original = pd.read_csv('data/ressources.csv', encoding='latin-1')

# Copie de travail — on ne touche JAMAIS à ressources_original
ressources = ressources_original.copy()

print(f" Dimensions : {ressources.shape[0]:,} lignes × {ressources.shape[1]} colonnes")
print(f" Agents uniques (Matricule) : {ressources['Matricule'].nunique():,}")
print()
print("Types de données :")
print(ressources.dtypes)

---
## Section 1 — Vue d'ensemble de la qualité des données

### 1.1 Valeurs uniques par colonne catégorielle

In [ ]:
colonnes_categorielles = ['Lieu.travail', 'Population', 'Site', 'Type.de.contrat']

for colonne in colonnes_categorielles:
    valeurs_uniques = ressources[colonne].value_counts()
    print(f"\n{colonne} ({ressources[colonne].nunique()} valeurs uniques) :")
    print(valeurs_uniques.to_string())

### 1.2 Tableau récapitulatif des anomalies

In [ ]:
def compter_anomalies_par_colonne(dataframe):
    """Calcule, pour chaque colonne, le nombre de NaN, de '???', et le total."""
    resultats = []
    for colonne in dataframe.columns:
        n_nan           = dataframe[colonne].isna().sum()
        n_interrogation = (dataframe[colonne].astype(str).str.strip() == '???').sum()
        total           = n_nan + n_interrogation
        pourcentage     = round(total / len(dataframe) * 100, 2)
        resultats.append({
            'Colonne'        : colonne,
            'NaN'            : n_nan,
            '???'            : n_interrogation,
            'Total anomalies': total,
            '% du total'     : pourcentage
        })
    return (pd.DataFrame(resultats)
              .sort_values('Total anomalies', ascending=False)
              .reset_index(drop=True))

tableau_anomalies_initial = compter_anomalies_par_colonne(ressources)
print("Tableau des anomalies initiales :")
tableau_anomalies_initial

---
## Section 2 — Anomalie B1 : Normalisation du format de `Date.presence`

### Constat
Toutes les dates sont au format français `DD/MM/YYYY`.

### Décision : Conversion vers le format ISO `YYYY/MM/DD`
> Cohérence avec les bases `dossier` et `temps` (format ISO).  
> Facilite les jointures temporelles et les comparaisons chronologiques sans conversion à la volée.

In [ ]:
# Aperçu du format actuel
print("Exemples de dates avant conversion :")
print(ressources['Date.presence'].head(5).tolist())
print()

def convertir_date_vers_iso(valeur_date_en_chaine):
    """Convertit DD/MM/YYYY vers YYYY/MM/DD. Retourne la valeur originale en cas d'échec."""
    try:
        return pd.to_datetime(valeur_date_en_chaine, dayfirst=True).strftime('%Y/%m/%d')
    except Exception:
        return valeur_date_en_chaine

valeurs_avant_conversion  = ressources['Date.presence'].copy()
ressources['Date.presence'] = ressources['Date.presence'].apply(convertir_date_vers_iso)
masque_dates_modifiees    = valeurs_avant_conversion != ressources['Date.presence']

print(f" {masque_dates_modifiees.sum():,} date(s) convertie(s) au format ISO")
print()
print("Exemples après conversion :")
print(ressources['Date.presence'].head(5).tolist())